# 🔬 Faultline — Node-by-Node Debug Notebook

Run each pipeline node **individually** and inspect its input/output at every step.

## Pipeline
```
ingest_signals
  → normalize_events
    → retrieve_related_situations
      → retrieve_calibration
        → map_situation
          → generate_predictions
            → map_market_implications
              → generate_actions
                → synthesize_report
                  → remember_situation
```

## Usage
- Run cells **top to bottom** — state accumulates as you go
- Each node cell shows its **inputs**, runs the node, then shows its **outputs**
- Change `SCENARIO_ID` or `PORTFOLIO_POSITIONS` / `WATCHLIST` at the top and re-run
- Skip any node to simulate a mid-pipeline failure

## ⚙️ Setup

In [ ]:
import sys, json, pprint
from pathlib import Path

# Ensure src/ is on the path when running without editable install
repo_root = Path().resolve().parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from dotenv import load_dotenv
load_dotenv(repo_root / ".env", override=False)

print("✅ Environment loaded")

In [ ]:
import uuid
from datetime import UTC, datetime
from pprint import pprint

from faultline.graph.workflow import StrategicSwarmWorkflow
from faultline.models import PortfolioPosition, WatchlistEntry
from faultline.persistence.store import SignalStore

# ── CONFIG ────────────────────────────────────────────────────────────────────
# Change these to explore different scenarios / portfolios
SCENARIO_ID  = "open_model_breakout"   # open_model_breakout | debt_defense_spiral | arctic_cable_bypass
RUN_MODE     = "demo"                   # demo | live
OUTPUT_DIR   = Path("/tmp/faultline_debug")

# Optional: add positions / watchlist to test portfolio-aware actions
PORTFOLIO_POSITIONS = [
    # PortfolioPosition(symbol="MSFT", direction="long", tags=["platform", "bundled"]),
]
WATCHLIST = [
    # WatchlistEntry(symbol="DDOG", tags=["open-source", "observability"]),
]
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
store = SignalStore(str(OUTPUT_DIR / "debug_runs.sqlite"))
wf    = StrategicSwarmWorkflow(store=store)

print(f"✅ Workflow ready  |  scenario={SCENARIO_ID}  mode={RUN_MODE}")

## 🛠 Display helpers

In [ ]:
from IPython.display import display, Markdown

def _json(obj):
    """Render a Pydantic model or plain dict as pretty JSON."""
    if hasattr(obj, "model_dump"):
        obj = obj.model_dump(mode="json")
    elif isinstance(obj, list):
        obj = [item.model_dump(mode="json") if hasattr(item, "model_dump") else item for item in obj]
    return json.dumps(obj, indent=2, default=str)

def show(label, value, max_items=5):
    """Print a labelled value — truncates long lists."""
    if isinstance(value, list) and len(value) > max_items:
        print(f"\n{'─'*60}")
        print(f"  {label}  [{len(value)} items — showing first {max_items}]")
        print('─'*60)
        for item in value[:max_items]:
            print(_json(item))
        print(f"  ... and {len(value)-max_items} more")
    else:
        print(f"\n{'─'*60}")
        print(f"  {label}")
        print('─'*60)
        print(_json(value) if not isinstance(value, (str, int, float, bool)) else value)

def diff_state(before: dict, after: dict):
    """Show only the keys that changed or were added."""
    changed = {k: v for k, v in after.items() if before.get(k) != v}
    if not changed:
        print("  (no state changes)")
    for k, v in changed.items():
        show(f"↳ state['{k}']", v)

print("✅ Display helpers ready")

## 🔄 Initialise state
This is the seed state passed into the first node.

In [ ]:
run_id = str(uuid.uuid4())

state = {
    "run_mode"            : RUN_MODE,
    "scenario_id"         : SCENARIO_ID,
    "portfolio_positions" : PORTFOLIO_POSITIONS,
    "watchlist"           : WATCHLIST,
    "diagnostics"         : {"run_id": run_id},
    "provenance"          : [],
}

print(f"Run ID : {run_id}")
print(f"Seed state keys: {list(state.keys())}")

---
## 📥 Node 1 — `ingest_signals`
**What it does:** Loads raw signals. In `demo` mode it pulls from sample providers. In `live` mode it calls real APIs.

**Inputs:** `run_mode`, `scenario_id`, `window_start` / `window_end` (live only)  
**Outputs:** `raw_signals`, `provenance`, `diagnostics.source_counts`

In [ ]:
print("═"*60)
print(" NODE: ingest_signals")
print("═"*60)

print("\n── INPUTS ──")
show("run_mode",   state.get("run_mode"))
show("scenario_id", state.get("scenario_id"))

before = dict(state)
out    = wf.ingest_signals(state)
state.update(out)

print("\n── OUTPUTS ──")
diff_state(before, state)

print(f"\n✅ {len(state.get('raw_signals', []))} raw signals ingested")

### Inspect individual raw signals

In [ ]:
# Change index to inspect a specific signal
IDX = 0
signals = state.get("raw_signals", [])
if signals:
    print(f"Signal [{IDX}] of {len(signals)}:")
    print(_json(signals[IDX]))
else:
    print("No signals — check run_mode / scenario_id")

---
## 🔀 Node 2 — `normalize_events`
**What it does:** Groups raw signals into `SignalEvent`s and clusters them by story key. Deduplicates against stored hashes (skipped in demo mode).

**Inputs:** `raw_signals`  
**Outputs:** `normalized_events`, `event_clusters`, `selected_cluster`, `diagnostics`

In [ ]:
print("═"*60)
print(" NODE: normalize_events")
print("═"*60)

print(f"\n── INPUT: {len(state.get('raw_signals', []))} raw signals ──")

before = dict(state)
out    = wf.normalize_events(state)
state.update(out)

print("\n── OUTPUTS ──")
diff_state(before, state)

clusters = state.get("event_clusters", [])
print(f"\n✅ {len(state.get('normalized_events', []))} events → {len(clusters)} clusters")
if clusters:
    sel = state["selected_cluster"]
    print(f"   Selected cluster: {sel.canonical_title} (strength={sel.cluster_strength:.2f})")

In [ ]:
# Explore all clusters
for i, c in enumerate(state.get("event_clusters", [])):
    print(f"[{i}] {c.canonical_title}  |  signals={len(c.signal_ids)}  strength={c.cluster_strength:.2f}  agreement={c.agreement_score:.2f}")

In [ ]:
# Override selected cluster if you want to debug a different one
# CLUSTER_IDX = 1
# state["selected_cluster"] = state["event_clusters"][CLUSTER_IDX]
# print(f"Overriding selected cluster → {state['selected_cluster'].canonical_title}")

---
## 🧠 Node 3 — `retrieve_related_situations`
**What it does:** Semantic search against the in-memory situation store to find prior analogous situations.

**Inputs:** `selected_cluster`  
**Outputs:** `related_situations`

In [ ]:
print("═"*60)
print(" NODE: retrieve_related_situations")
print("═"*60)

cluster = state.get("selected_cluster")
print(f"\n── INPUT: cluster = '{cluster.canonical_title if cluster else None}' ──")

before = dict(state)
out    = wf.retrieve_related_situations(state)
state.update(out)

print("\n── OUTPUTS ──")
related = state.get("related_situations", [])
print(f"  related_situations: {len(related)} found")
for r in related:
    show(r.title, r)

---
## 📊 Node 4 — `retrieve_calibration`
**What it does:** Loads historical calibration signals (confirmed/unconfirmed prediction rates) from the SQLite store. Used to adjust confidence scores downstream.

**Inputs:** `diagnostics.run_id`  
**Outputs:** `calibration_signals`

In [ ]:
print("═"*60)
print(" NODE: retrieve_calibration")
print("═"*60)

print(f"\n── INPUT: run_id = {state['diagnostics'].get('run_id')} ──")

before = dict(state)
out    = wf.retrieve_calibration(state)
state.update(out)

sigs = state.get("calibration_signals", [])
print(f"\n── OUTPUT: {len(sigs)} calibration signals ──")
for cs in sigs:
    show(f"{cs.prediction_type}", cs)

---
## 🗺 Node 5 — `map_situation`
**What it does:** Builds the `SituationSnapshot` — actors, mechanisms, forces, stage assessment, confidence. This is the core structural model of the geopolitical situation.

**Inputs:** `selected_cluster`, `normalized_events`, `related_situations`  
**Outputs:** `situation_snapshot`

In [ ]:
print("═"*60)
print(" NODE: map_situation")
print("═"*60)

cluster = state.get("selected_cluster")
print(f"\n── INPUTS ──")
print(f"  cluster          : {cluster.canonical_title if cluster else None}")
print(f"  normalized_events: {len(state.get('normalized_events', []))}")
print(f"  related_situations: {len(state.get('related_situations', []))}")

before = dict(state)
out    = wf.map_situation(state)
state.update(out)

snap = state.get("situation_snapshot")
print("\n── OUTPUT: situation_snapshot ──")
if snap:
    print(f"  title        : {snap.title}")
    print(f"  system       : {snap.system_under_pressure}")
    print(f"  stage        : {snap.stage.stage}  (confidence={snap.confidence:.2f})")
    print(f"  actors       : {[a.name for a in snap.key_actors]}")
    print(f"  mechanisms   : {[m.name for m in snap.mechanisms]}")
    print(f"  forces       : {len(snap.forces)}")
    print("\n  Full snapshot:")
    print(_json(snap))

---
## 🔮 Node 6 — `generate_predictions`
**What it does:** Generates actor-move predictions, a scenario tree (base / acceleration / reversal branches), and stage-transition warnings.

**Inputs:** `situation_snapshot`, `selected_cluster`, `calibration_signals`  
**Outputs:** `predictions`, `scenario_tree`, `stage_transition_warnings`

In [ ]:
print("═"*60)
print(" NODE: generate_predictions")
print("═"*60)

snap = state.get("situation_snapshot")
print(f"\n── INPUTS ──")
print(f"  snapshot stage       : {snap.stage.stage if snap else None}")
print(f"  snapshot confidence  : {snap.confidence if snap else None}")
print(f"  calibration signals  : {len(state.get('calibration_signals', []))}")

before = dict(state)
out    = wf.generate_predictions(state)
state.update(out)

preds    = state.get("predictions", [])
branches = state.get("scenario_tree", [])
warnings = state.get("stage_transition_warnings", [])

print(f"\n── OUTPUTS ──")
print(f"  predictions           : {len(preds)}")
print(f"  scenario_tree branches: {len(branches)}")
print(f"  stage warnings        : {len(warnings)}")

print("\n  Predictions:")
for p in preds:
    print(f"  [{p.prediction_type}] conf={p.confidence:.2f}  horizon={p.time_horizon}")
    print(f"    {p.description}")

print("\n  Scenario tree:")
for b in branches:
    print(f"  [{b.scenario_type}] prob={b.probability:.2f}  {b.label}")

if warnings:
    print("\n  Stage warnings:")
    for w in warnings:
        print(f"  ⚠ {w.current_stage} → {w.next_stage}  prob={w.probability:.2f}  lead={w.lead_time}")

In [ ]:
# Deep-inspect a specific prediction
IDX = 0
if state.get("predictions"):
    print(_json(state["predictions"][IDX]))

---
## 💹 Node 7 — `map_market_implications`
**What it does:** Translates the structural situation into market implications — asset repricing, high-confidence opportunities, thematic tilts.

**Inputs:** `situation_snapshot`, `predictions`, `selected_cluster`, `calibration_signals`  
**Outputs:** `market_implications`

In [ ]:
print("═"*60)
print(" NODE: map_market_implications")
print("═"*60)

print(f"\n── INPUTS ──")
print(f"  predictions      : {len(state.get('predictions', []))}")
print(f"  calibration sigs : {len(state.get('calibration_signals', []))}")

before = dict(state)
out    = wf.map_market_implications(state)
state.update(out)

implications = state.get("market_implications", [])
print(f"\n── OUTPUT: {len(implications)} implications ──")
for imp in implications:
    print(f"  [{imp.thesis_type}] {imp.direction.upper()}  conf={imp.confidence:.2f}  target={imp.target}")
    print(f"    {imp.rationale}")

In [ ]:
# Deep-inspect an implication
IDX = 0
if state.get("market_implications"):
    print(_json(state["market_implications"][IDX]))

---
## ⚡ Node 8 — `generate_actions`
**What it does:** Generates action recommendations (enter / exit / trim / avoid / watch) for direct positions, watchlist entries, and structural signals. Applies `OperatorPolicyConfig` thresholds.

**Inputs:** `situation_snapshot`, `market_implications`, `predictions`, `calibration_signals`, `portfolio_positions`, `watchlist`, `stage_transition_warnings`, `operator_policy_config`  
**Outputs:** `action_recommendations`, `exit_signals`, `endangered_symbols`

In [ ]:
# Optionally add portfolio positions here to test portfolio-aware actions
# from faultline.models import PortfolioPosition, WatchlistEntry
# state["portfolio_positions"] = [
#     PortfolioPosition(symbol="MSFT", direction="long", tags=["platform", "bundled"]),
# ]
# state["watchlist"] = [
#     WatchlistEntry(symbol="DDOG", tags=["open-source", "observability"]),
# ]

In [ ]:
print("═"*60)
print(" NODE: generate_actions")
print("═"*60)

print(f"\n── INPUTS ──")
print(f"  market_implications    : {len(state.get('market_implications', []))}")
print(f"  predictions            : {len(state.get('predictions', []))}")
print(f"  portfolio_positions    : {len(state.get('portfolio_positions', []))}")
print(f"  watchlist              : {len(state.get('watchlist', []))}")
print(f"  stage_warnings         : {len(state.get('stage_transition_warnings', []))}")
print(f"  operator_policy_config : {state.get('operator_policy_config')}")

before = dict(state)
out    = wf.generate_actions(state)
state.update(out)

actions   = state.get("action_recommendations", [])
exits     = state.get("exit_signals", [])
endangered = state.get("endangered_symbols", [])

print(f"\n── OUTPUTS ──")
print(f"  action_recommendations : {len(actions)}")
print(f"  exit_signals           : {len(exits)}")
print(f"  endangered_symbols     : {endangered}")

print("\n  All actions:")
for a in sorted(actions, key=lambda x: x.confidence, reverse=True):
    print(f"  [{a.action.upper():5}] conf={a.confidence:.2f} urgency={a.urgency:6}  {a.target}")
    print(f"         {a.rationale[:90]}")

In [ ]:
# Deep-inspect a specific action
IDX = 0
if state.get("action_recommendations"):
    print(_json(state["action_recommendations"][IDX]))

---
## 📝 Node 9 — `synthesize_report`
**What it does:** Assembles the final `StructuredReport` — narrative, scenario tree, confidence boundaries, action traceability, publication decision.

**Inputs:** `situation_snapshot`, `selected_cluster`, `related_situations`, `calibration_signals`, `predictions`, `scenario_tree`, `stage_transition_warnings`, `market_implications`, `action_recommendations`, `exit_signals`, `endangered_symbols`, `provenance`  
**Outputs:** `final_report`, `diagnostics.publish_decision`

In [ ]:
print("═"*60)
print(" NODE: synthesize_report")
print("═"*60)

print(f"\n── INPUTS (summary) ──")
for key in ["predictions", "scenario_tree", "market_implications", "action_recommendations", "exit_signals"]:
    print(f"  {key:30}: {len(state.get(key, []))}")

before = dict(state)
out    = wf.synthesize_report(state)
state.update(out)

report = state.get("final_report")
print(f"\n── OUTPUT ──")
print(f"  publication_status : {report.publication_status if report else None}")
print(f"  narrative length   : {len(report.narrative) if report and report.narrative else 0} chars")
print(f"  scenario_tree      : {len(report.scenario_tree) if report else 0} lines")
print(f"  confidence_bounds  : {len(report.confidence_boundaries) if report else 0} items")
print(f"  action_traceability: {len(report.action_traceability) if report else 0} items")

In [ ]:
# Render the full report as Markdown
from faultline.synthesis.report_builder import render_markdown

if state.get("final_report"):
    display(Markdown(render_markdown(state["final_report"])))

---
## 💾 Node 10 — `remember_situation`
**What it does:** Persists the `SituationSnapshot` into the in-memory semantic store (for future `retrieve_related_situations` calls) and saves it to SQLite.

**Inputs:** `situation_snapshot`  
**Outputs:** _(updates memory and DB, returns empty dict)_

In [ ]:
print("═"*60)
print(" NODE: remember_situation")
print("═"*60)

snap = state.get("situation_snapshot")
print(f"\n── INPUT: snapshot = '{snap.title if snap else None}' ──")

out = wf.remember_situation(state)
state.update(out)

print("  ✅ Situation persisted to memory + SQLite")

---
## 🔍 Full State Inspector
See everything in the accumulated state after running all nodes.

In [ ]:
print(f"Final state keys ({len(state)}):")
for k, v in state.items():
    if isinstance(v, list):
        print(f"  {k:35} → list[{len(v)}]")
    elif hasattr(v, '__class__'):
        print(f"  {k:35} → {type(v).__name__}")
    else:
        print(f"  {k:35} → {v!r}")

In [ ]:
# Dump any single state key as JSON for deep inspection
KEY = "situation_snapshot"  # change to any key above
if state.get(KEY):
    print(_json(state[KEY]))
else:
    print(f"{KEY!r} not in state")

---
## ♻️ Re-run a single node with modified input
Modify part of the state and re-run just one node to see how it responds.

In [ ]:
# Example: manually raise snapshot confidence and re-run generate_actions
from copy import deepcopy
from faultline.models import OperatorPolicyConfig

patched_state = deepcopy(state)

# Lower the implication confidence threshold to get more aggressive actions
patched_state["operator_policy_config"] = OperatorPolicyConfig(
    implication_enter_threshold=0.55,
    watchlist_enter_threshold=0.55,
    timing_trim_threshold=0.45,
    timing_exit_threshold=0.65,
)

rerun_out = wf.generate_actions(patched_state)
patched_actions = rerun_out.get("action_recommendations", [])

print(f"Original actions : {len(state.get('action_recommendations', []))}")
print(f"With lower thresh: {len(patched_actions)}")
print("\nNew / additional actions:")
orig_targets = {a.target for a in state.get("action_recommendations", [])}
for a in patched_actions:
    marker = "  " if a.target in orig_targets else "🆕"
    print(f"{marker} [{a.action.upper():5}] {a.target}  conf={a.confidence:.2f}")

---
## 🚀 Run the full pipeline at once (for comparison)
Compare against the node-by-node run above.

In [ ]:
from faultline.graph.runner import StrategicSwarmRunner

runner = StrategicSwarmRunner(
    output_dir=OUTPUT_DIR / "full_run",
    database_url=f"sqlite:///{OUTPUT_DIR / 'full_run.sqlite'}",
)
full_result = runner.run_demo(SCENARIO_ID)

fs = full_result["final_state"]
print(f"Full pipeline result:")
print(f"  predictions         : {len(fs.get('predictions', []))}")
print(f"  market_implications : {len(fs.get('market_implications', []))}")
print(f"  action_recommendations: {len(fs.get('action_recommendations', []))}")
print(f"  publication_status  : {fs['final_report'].publication_status if fs.get('final_report') else None}")
display(Markdown(render_markdown(fs["final_report"])))